In [1]:
!pip install boto3 awscli mlflow

In [2]:
import setup_up
import os 
print(os.getenv("AWS_DEFAULT_REGION"))
print(bool(os.getenv("AWS_ACCESS_KEY_ID")))
print(bool(os.getenv("AWS_SECRET_ACCESS_KEY")))

eu-north-1
True
True


In [3]:
!aws sts get-caller-identity > /dev/null

In [4]:
import mlflow
import setup_up
SERVER_URL = os.environ["SERVER_URL"]

mlflow.set_tracking_uri(SERVER_URL)

In [5]:
mlflow.set_experiment('Exp 3 - TfIdf Trigram max_features')

/home/grayfog/miniconda3/envs/YTsentimentAnalyzer/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
2026/03/14 23:47:40 INFO mlflow.tracking.fluent: Experiment with name 'Exp 3 - TfIdf Trigram max_features' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/grayfog/YTSentiment/notebooks/ec2-13-51-197-16.eu-north-1.compute.amazonaws.com/397579181591453453', creation_time=1773528460824, experiment_id='397579181591453453', last_update_time=1773528460824, lifecycle_stage='active', name='Exp 3 - TfIdf Trigram max_features', tags={}>

In [6]:
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
df = pd.read_csv('../data/read_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 3)

In [9]:
def run_experiment_tfidf_max_features(max_features):
    ngram_range = (1,3) # trigram

    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)
    
    mlflow.end_run()
    with mlflow.start_run() as run:
        mlflow.set_tag('mlflow.runName', f'TFIDF_Trigrams_features_{max_features}')
        mlflow.set_tag('experiment_type', 'feature_engineering')
        mlflow.set_tag('model_type', 'RandomForestClassifier')

        mlflow.set_tag('descritption', f'RandomForest with TD-IDF Trigrams, max_feature = {max_feature}')

        mlflow.log_param('vectorize_type','TF-IDF')
        mlflow.log_param('ngram_range', ngram_range)
        mlflow.log_param('vectorizer_max_features', max_features)

        n_estimators = 200
        max_depth = 15
        
        mlflow.log_param('n_estimators', n_estimators)
        mlflow.log_param('max_depth', max_depth)

        # initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,random_state=42)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
    
        # log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)
    
        # log classification report 
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics,dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f'{label}_{metric}', value)

        conf_matrix = confusion_matrix(y_test,y_pred)
        plt.figure(figsize=(8,6))
        sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
        plt.xlabel = 'Predicted'
        plt.ylable = 'Actual'
        plt.title = f'Confusion Matrix:TD-IDF_max_features={max_features}'
        plt.savefig('../data/confusion_matrix_exp4.png')
        mlflow.log_artifact('../data/confusion_matrix_exp4.png')
        plt.close()
    
        # log the model
        mlflow.sklearn.log_model(model, name=f'random_forest_model_TFIDF_{max_features}')

max_features_values = [1000,2000,3000,4000,5000,6000,7000,8000,9000,10000]

for max_feature in max_features_values:
    run_experiment_tfidf_max_features(max_feature)

/home/grayfog/miniconda3/envs/YTsentimentAnalyzer/lib/python3.12/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)
/home/grayfog/miniconda3/envs/YTsentimentAnalyzer/lib/python3.12/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)
/home/grayfog/miniconda3/envs/YTsentimentAnalyzer/lib/pyth